# Regular Expressions

A regular expression (shortened as regex or regexp), sometimes referred to as a rational expression, is a sequence of characters that specifies a match pattern in text. Usually such patterns are used by string-searching algorithms for "find" or "find and replace" operations on strings, or for input validation. Regular expression techniques are developed in theoretical computer science and formal language theory.

In [11]:
import re
from typing import Self, Optional
from dataclasses import dataclass, field
import colorama

In [12]:
res = re.search("seminars?k?i?","seminar")
if res:
    print("Found:")
    print(f"Raw: {res}")
    print(f"Group: {res.group()}")
else:
    print("Not found")

Found:
Raw: <re.Match object; span=(0, 7), match='seminar'>
Group: seminar


# Types of Regex engines
There are two types of Regex engines:
1. Regex-directed engines ( NFA engines )
2. Text-directed engines  ( DFA engines )

There are pros and cons to each model, the main ones being that NFA engines support more complex patterns, but can run in O(2^n) in worst case scenarios, and that DFA is strictly linear in execution time (O(n)).

In this project, we will implement a Regex-directed engine.


In [13]:
class StateContext():
    text = None
    index = 0
    captured_groups = []
    
    def __init__(self,text,index=0,captured_groups=[]):
        self.text = text
        self.index = index
        self.captured_groups = captured_groups
        pass

In [14]:
@dataclass
class Node():
    _type = "" # Literal (leaf) or Operator (applied to children)
    def __init__(self,val):
        self.val = val
        self.children = []
        
    def accept(self,visitor):
        method_name = f'visit_{self.__class__.__name__.lower()}'
        visitor_method = getattr(visitor,method_name,visitor.generic_visit())
        return visitor_method(self)
    
    def match(self, ctx : StateContext, node : Self):
        pass

@dataclass
class Literal(Node):
    char : str
    
    def match(self,ctx : StateContext) -> bool:
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] == self.char:
            ctx.index+=1
            return True
        return False

@dataclass
class Concatenation(Node):
    children : list[Node]
    
    def match(self, ctx : StateContext) -> bool:
        start_index = ctx.index
        for child in self.children:
            if not child.match(ctx):
                ctx = start_index
                return False
        return True
    
@dataclass
class Alternation(Node):
    left : Node
    right : Node
    
    def match(self, ctx: StateContext) -> bool:
        start_index = ctx.index
        
        if self.left.match(ctx):
            return True
        else:
            ctx.index = start_index
            return self.right.match(ctx)

@dataclass
class Wildcard(Node):
    def match(self, ctx: StateContext) -> bool:
        if ctx.index < len(ctx.text) and ctx.text[ctx.index]!='\n':
            ctx.index += 1
            return True
        return False

@dataclass
class Quantifier(Node):
    child : Node
    min_repeat : int
    max_repeat : Optional[int]
    
@dataclass
class Group(Node):
    child : Node
    index : int
    is_capturing : bool

In [15]:
class Tester():
    tests: dict[str, list[tuple[Node,str, bool]]]
    
    def __init__(self,raw_tests):
        self.tests = {}
        for rule,tree,string,expected in raw_tests:
            if rule in self.tests.keys():
                self.tests[rule].append((tree,string,expected))
            else:
                self.tests[rule] = [(tree,string,expected)]
        pass
    
    def test(self):
        res = []
        i = 0
        for rule_name, tests in self.tests.items():
            for (tree,string,answer) in tests:
                if tree.match(StateContext(string)) == answer:
                    # Number, expected, Correct, String
                    res.append([i,answer,True, string])
                else:
                    res.append([i,not answer,False, string])
                i+=1
        
        for r in res:
            print(f"Test {r[0]}| Pass: {colorama.Fore.GREEN if r[2] else colorama.Fore.RED}{'PASS' if r[2] else 'FAIL' }{colorama.Style.RESET_ALL}| Str: {r[3]}| Res: {r[1]} |")
        

In [16]:
class Lexer():
    # Parses the input string into tokens
    def __init__(self):
        pass

In [17]:
class Parser():
    # Turns tokens into an AST object hierarchy.
    def __init__(self):
        pass

In [ ]:
state = StateContext("abc")

abc_tree = Concatenation(children=[Literal("a"),
                              Literal("b"),
                              Wildcard(),
                              Wildcard(),
                              Wildcard(),
                              Literal("f")])

tests = [["ab...f",
           abc_tree,
           "abxxxc",
           False
         ],
         ["ab...f",
          abc_tree,
          "abxxxf",
          True
          ],
         ["ab...f",
          abc_tree,
          "abcf",
          False
          ],
         ]

tester = Tester(tests)
tester.test()

Test 0| Pass: PASS| Str: abxxxc| Res: False |
Test 1| Pass: PASS| Str: abxxxf| Res: True |
Test 2| Pass: FAIL| Str: abcf| Res: False |
